# Part 1 - K-Nearest Neighbors (KNN)
**Gray Interface '26 | Task 3**

Dataset: [Wine Quality](https://www.kaggle.com/datasets/sinadehestani/wine-quality-red-and-white)

## Importing the dependencies

In [ ]:
!pip install kagglehub --quiet


In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


## Loading the Dataset

In [ ]:
path = kagglehub.dataset_download("sinadehestani/wine-quality-red-and-white")
print(os.listdir(path))


In [ ]:
# Load both red and white wine CSVs and combine them
red   = pd.read_csv(os.path.join(path, "winequality-red.csv"),   sep=';')
white = pd.read_csv(os.path.join(path, "winequality-white.csv"), sep=';')

red['wine_type']   = 'red'
white['wine_type'] = 'white'

df = pd.concat([red, white], ignore_index=True)
print("Shape:", df.shape)
df.head()


## Exploratory Data Analysis

In [ ]:
print(df.info())
print("\nMissing values:", df.isnull().sum().sum())


In [ ]:
# Distribution of wine quality scores
plt.figure(figsize=(8, 5))
df['quality'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Distribution of Wine Quality Scores')
plt.xlabel('Quality')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(11, 8))
sns.heatmap(df.drop(columns=['wine_type']).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()


## Preprocessing

In [ ]:
# Encode wine_type: red=0, white=1
df['wine_type'] = LabelEncoder().fit_transform(df['wine_type'])

# Convert quality into binary classification: good (>=7) vs not good (<7)
# This makes the problem cleaner and more practical
df['label'] = (df['quality'] >= 7).astype(int)
print("Class distribution:")
print(df['label'].value_counts())

X = df.drop(columns=['quality', 'label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


## Effect of Different Scaling Techniques

In [ ]:
# KNN uses distance — scaling matters a lot
# Features with larger ranges dominate distance calculations without scaling

scalers = {
    'No Scaling':    None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler':   MinMaxScaler(),
}

scaling_results = []
for name, scaler in scalers.items():
    if scaler:
        X_tr = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)
    else:
        X_tr, X_te = X_train.values, X_test.values

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_tr, y_train)
    acc = accuracy_score(y_test, knn.predict(X_te))
    f1  = f1_score(y_test, knn.predict(X_te))
    scaling_results.append({'Scaler': name, 'Accuracy': round(acc, 4), 'F1': round(f1, 4)})

pd.DataFrame(scaling_results)


## Effect of Different Distance Metrics

In [ ]:
# Use StandardScaler for all metric experiments (best scaler from above)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

metrics_results = []
for metric in ['euclidean', 'manhattan', 'minkowski']:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn.fit(X_tr, y_train)
    acc = accuracy_score(y_test, knn.predict(X_te))
    f1  = f1_score(y_test, knn.predict(X_te))
    metrics_results.append({'Metric': metric, 'Accuracy': round(acc, 4), 'F1': round(f1, 4)})

pd.DataFrame(metrics_results)


## Effect of Weighting Methods

In [ ]:
# 'uniform': all neighbors vote equally
# 'distance': closer neighbors have more influence
weight_results = []
for weights in ['uniform', 'distance']:
    knn = KNeighborsClassifier(n_neighbors=5, weights=weights)
    knn.fit(X_tr, y_train)
    acc = accuracy_score(y_test, knn.predict(X_te))
    f1  = f1_score(y_test, knn.predict(X_te))
    weight_results.append({'Weights': weights, 'Accuracy': round(acc, 4), 'F1': round(f1, 4)})

pd.DataFrame(weight_results)


## Accuracy vs K and F1 vs K

In [ ]:
k_values = list(range(1, 31))
acc_list, f1_list = [], []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='euclidean')
    knn.fit(X_tr, y_train)
    preds = knn.predict(X_te)
    acc_list.append(accuracy_score(y_test, preds))
    f1_list.append(f1_score(y_test, preds))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(k_values, acc_list, marker='o', color='steelblue')
ax1.set_title('Accuracy vs K')
ax1.set_xlabel('K')
ax1.set_ylabel('Accuracy')

ax2.plot(k_values, f1_list, marker='o', color='#E50914')
ax2.set_title('F1-Score vs K')
ax2.set_xlabel('K')
ax2.set_ylabel('F1 Score')

plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(f1_list)]
print(f"Best K by F1: {best_k}  (F1 = {max(f1_list):.4f})")


## Best Model — Final Evaluation

In [ ]:
best_knn = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
best_knn.fit(X_tr, y_train)
preds = best_knn.predict(X_te)

print(classification_report(y_test, preds, target_names=['Not Good', 'Good']))

# Confusion matrix
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['True: 0', 'True: 1'])
plt.title(f'Confusion Matrix (K={best_k})')
plt.tight_layout()
plt.show()


## Observations

- **No Scaling** gives noticeably worse results — features like `total sulfur dioxide` (range 0–300) dominate over `pH` (range 2.8–3.8), making distances meaningless.
- **StandardScaler** and **MinMaxScaler** both improve performance; StandardScaler tends to work slightly better for KNN on this dataset.
- **Distance weighting** outperforms uniform weighting — closer neighbors are more informative.
- **Euclidean and Minkowski (p=2)** are equivalent and perform well; Manhattan works similarly.
- The Accuracy vs K and F1 vs K plots both show the sweet spot is usually between K=7 and K=15 — very low K overfits (jagged boundary), very high K underfits (too smooth).
